# 04 — Model Architectures

Inspect and smoke-test each model on one synthetic batch. This verifies the tensor contract without training or changing checkpoints.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

import torch
from models import (
    PersistenceModel,
    TemporalLSTM,
    TemporalTransformer,
    GraphAttentionLayer,
    SpatiotemporalGATTransformer,
)


In [ ]:
batch_size = 4
lookback = 24
stations = 12
features = 5
x = torch.randn(batch_size, lookback, stations, features)
adjacency = torch.ones(stations, stations)

models = {
    "Persistence": (PersistenceModel(), lambda model: model.predict(x)),
    "LSTM": (TemporalLSTM().eval(), lambda model: model(x)),
    "Transformer": (TemporalTransformer().eval(), lambda model: model(x)),
    "GAT-Transformer": (SpatiotemporalGATTransformer().eval(), lambda model: model(x, adjacency)),
}

with torch.no_grad():
    for name, (model, forward) in models.items():
        output = forward(model)
        print(f"{name}: output={tuple(output.shape)}, parameters={sum(p.numel() for p in model.parameters()) if hasattr(model, 'parameters') else 'n/a'}")


In [ ]:
gat = GraphAttentionLayer(in_features=features, out_features=features + 1)
h, attention = gat(torch.randn(batch_size, stations, features), adjacency)
print("GAT node output:", tuple(h.shape))
print("GAT attention:", tuple(attention.shape))
